# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Gradient-boosted trees (`HistGradientBoostingRegressor`), predicting `ctr_gap_last30`.**

Why this method for this lane:
- The signal audit (ML-06) found the CTR-gap signal is real but the relationship with position
  is non-linear and has heavy tails (a few high-impression pages dominate). Tree-based models
  handle non-linearity and outliers without manual transforms.
- ML-05's feature set mixes types that trees handle natively without much preprocessing: counts
  (`backlinks`), rates (`competition`), and categoricals with missing values (`_was_missing`
  flags already built in ML-05).
- It's still inspectable — feature importances give a plain-language "what is it leaning on"
  answer for Section 4, unlike a black-box deep model this dataset size doesn't need anyway.

**What the model is actually asked to do, precisely (resolving the open question from ML-03):**
predict `ctr_gap_last30` using *only* features available **before** the `last30` window —
`prev30` performance + static SEO/content metadata. It never sees this month's own
impressions/clicks/position, the same numbers the baseline (ML-07) uses directly. That keeps
the comparison in Section 3 honest: both the model and its baseline counterpart here are working
from the same, earlier information, predicting the same later outcome.


In [8]:
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, KFold


# ============================================================
# 1. Load data
# ============================================================

DATA_DIR = "/home/mahad/projects/flyrank-ml-internship-starter/work/data"

fact = pd.read_parquet(
    f"{DATA_DIR}/fact_content_query_90d.parquet"
)

dc = pd.read_parquet(
    f"{DATA_DIR}/dim_content.parquet"
)


# ============================================================
# 1A. Keep only rows with actual last-30-day click activity
# ============================================================

fact = fact[fact["clicks_last30"] > 0].copy()

print("Filtered fact rows:", len(fact))
print("Unique clients:", fact["client_hash_id"].nunique())


# ============================================================
# 2. Create target: ctr_gap_last30
# ============================================================

fact["ctr_last30"] = (
    fact["clicks_last30"] /
    fact["impressions_last30"].replace(0, np.nan)
)

bins = [0, 3, 5, 10, 20, 50, 1000]
pos_labels = ["1-3", "3-5", "5-10", "10-20", "20-50", "50+"]

fact["pos_bucket_last30"] = pd.cut(
    fact["avg_position_last30"],
    bins=bins,
    labels=pos_labels
).astype(str)

exp_last30 = fact.groupby(
    "pos_bucket_last30"
).apply(
    lambda g: g["clicks_last30"].sum() /
              g["impressions_last30"].sum()
)

fact["ctr_gap_last30"] = (
    fact["pos_bucket_last30"]
    .map(exp_last30)
    .astype(float)
    - fact["ctr_last30"]
)


# ============================================================
# 3. Define features
# ============================================================

FACT_FEATURES = [
    "query_char_count",
    "query_token_count",
    "impressions_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_visible_query_count",
    "rare_query_count",
    "rare_impressions_share"
]

DC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "backlinks",
    "char_count",
    "word_count"
]


# ============================================================
# 4. Keep live content
# ============================================================

dc_live = dc[
    dc["is_published"] & ~dc["is_deleted"]
][
    ["client_hash_id", "content_hash_id"] + DC_FEATURES
]


# ============================================================
# 5. Build modeling dataframe
# ============================================================

df = fact[
    [
        "client_hash_id",
        "content_hash_id",
        "query_hash_id",
        "pos_bucket_last30"
    ]
    + FACT_FEATURES
    + ["ctr_gap_last30"]
].merge(
    dc_live,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
).dropna(
    subset=["ctr_gap_last30"]
)


# ============================================================
# 6. Create X, y, groups
# ============================================================

X = df[FACT_FEATURES + DC_FEATURES].copy()

for col in X.columns:
    if X[col].isnull().any():
        X[col + "_was_missing"] = X[col].isnull().astype(int)
        X[col] = X[col].fillna(X[col].median())

y = df["ctr_gap_last30"].copy()

groups = df["client_hash_id"]


print("X:", X.shape)
print("y:", y.shape)
print("unique clients:", groups.nunique())

Filtered fact rows: 85115
Unique clients: 41
X: (83310, 21)
y: (83310,)
unique clients: 41


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_hash_id`, not random, not time-based (this data is a single fixed
snapshot, so there's no later time slice to split on).**

Random row-level splitting would put some queries from a given client in train and others from
the *same client* in test. A client's overall SEO practices (site structure, domain authority,
content strategy) are a hidden factor that correlates across all of that client's rows — a
random split would let the model partly "memorize" a client's baseline quality from train and
just apply it to that client's test rows, inflating the score without proving the model
generalizes to a client it's never seen. `GroupKFold` on `client_hash_id` forces every fold to
be evaluated on entirely unseen clients — an honest test of "does this work for a new client,"
which is the real use case (FlyRank's clients aren't in the training set when the product
ships).


In [9]:
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X, y, groups=groups))
print("Folds:", len(splits))
for i, (tr, te) in enumerate(splits):
    print(f"Fold {i}: train clients={groups.iloc[tr].nunique()}, test clients={groups.iloc[te].nunique()}, "
          f"overlap={len(set(groups.iloc[tr]) & set(groups.iloc[te]))}")


Folds: 5
Fold 0: train clients=40, test clients=1, overlap=0
Fold 1: train clients=31, test clients=10, overlap=0
Fold 2: train clients=32, test clients=9, overlap=0
Fold 3: train clients=30, test clients=11, overlap=0
Fold 4: train clients=31, test clients=10, overlap=0


In [10]:
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X, y, groups=groups))

print("Folds:", len(splits))

for i, (tr, te) in enumerate(splits):
    print(
        f"Fold {i}: "
        f"train clients={groups.iloc[tr].nunique()}, "
        f"test clients={groups.iloc[te].nunique()}, "
        f"overlap={len(set(groups.iloc[tr]) & set(groups.iloc[te]))}, "
        f"test rows={len(te)}"
    )

Folds: 5
Fold 0: train clients=40, test clients=1, overlap=0, test rows=20256
Fold 1: train clients=31, test clients=10, overlap=0, test rows=15766
Fold 2: train clients=32, test clients=9, overlap=0, test rows=15763
Fold 3: train clients=30, test clients=11, overlap=0, test rows=15763
Fold 4: train clients=31, test clients=10, overlap=0, test rows=15762


In [11]:
print("\n========== TARGET TIE CHECK ==========")

most_common_count = y.value_counts().iloc[0]

print("y length:", len(y))
print("unique y values:", y.nunique())
print("most common y value:", most_common_count)

print(
    "Fraction of y equal to most common value:",
    most_common_count / len(y)
)

print("\nTop 10 most frequent y values:")
print(y.value_counts().head(10))


========== TARGET TIE CHECK ==========
y length: 83310
unique y values: 10115
most common y value: 540
Fraction of y equal to most common value: 0.006481814908174289

Top 10 most frequent y values:
ctr_gap_last30
-0.094598    540
-0.077931    483
-0.085507    461
-0.071521    458
-0.088587    453
-0.066026    445
-0.131444    435
-0.113587    433
-0.079496    430
-0.105709    429
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same data, same metric, same split as the baseline — the baseline is redefined slightly to
make that a fair fight (explained below), then both are scored the same way: Precision@50.**

ML-07's baseline computes `ctr_gap` from *this month's own* real numbers — it isn't predicting
anything, so it can't be compared apples-to-apples to a model that only sees `prev30`. For this
comparison, the baseline gets the same information handicap as the model: **`prev30_ctr_gap`**
— the identical rule (expected CTR for position bucket, minus actual CTR), computed on `prev30`
data instead of `last30`. Both the model and this baseline rank pages using only pre-`last30`
information; both get checked against the same real, already-known `ctr_gap_last30`.

**Precision@50:** of the top 50 pages each method ranks highest, what fraction land in the
*actual* top 50 by real `ctr_gap_last30`. Run the cell below and put the real two numbers here —
I have not written a winner in this paragraph on purpose.


In [12]:
# Baseline: the same rule, computed one window earlier (prev30), so it's not cheating with
# information the model doesn't get either.
fact["pos_bucket_prev30"] = pd.cut(fact.avg_position_prev30, bins=bins, labels=pos_labels).astype(str)
exp_prev30 = fact.groupby("pos_bucket_prev30").apply(
    lambda g: g.clicks_prev30.sum() / g.impressions_prev30.replace(0, np.nan).sum())
fact["ctr_prev30"] = fact.clicks_prev30 / fact.impressions_prev30.replace(0, np.nan)
fact["prev30_ctr_gap"] = fact.pos_bucket_prev30.map(exp_prev30).astype(float) - fact.ctr_prev30

# Bring prev30_ctr_gap onto df by the same join keys used everywhere else — no index-matching guesswork
df = df.merge(
    fact[["client_hash_id", "content_hash_id", "query_hash_id", "prev30_ctr_gap"]],
    on=["client_hash_id", "content_hash_id", "query_hash_id"], how="left"
)

def precision_at_k(scores, y_true, k=50):
    scores = pd.Series(scores).reset_index(drop=True)
    y_true = pd.Series(y_true).reset_index(drop=True)
    top_pred = set(scores.sort_values(ascending=False).head(k).index)
    top_true = set(y_true.sort_values(ascending=False).head(k).index)
    return len(top_pred & top_true) / k

model_precisions, baseline_precisions = [], []
for tr, te in splits:
    model = HistGradientBoostingRegressor(random_state=42)
    model.fit(X.iloc[tr], y.iloc[tr])
    pred = model.predict(X.iloc[te])
    y_te = y.iloc[te]
    baseline_te = df["prev30_ctr_gap"].iloc[te]

    model_precisions.append(precision_at_k(pred, y_te))
    baseline_precisions.append(precision_at_k(baseline_te, y_te))

results = pd.DataFrame({"fold": range(len(splits)),
                         "model_precision_at_50": model_precisions,
                         "baseline_precision_at_50": baseline_precisions})
print(results)
print()
print("Mean model Precision@50:", results.model_precision_at_50.mean())
print("Mean baseline Precision@50:", results.baseline_precision_at_50.mean())
print()

   fold  model_precision_at_50  baseline_precision_at_50
0     0                   0.00                       0.0
1     1                   0.08                       0.0
2     2                   0.02                       0.0
3     3                   0.06                       0.0
4     4                   0.02                       0.0

Mean model Precision@50: 0.036
Mean baseline Precision@50: 0.0



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Do not fill this in until the cell below has actually run — feature importances and error
buckets are exactly the kind of number that's easy to guess wrong.** What to look for once it
runs: (1) which features the model actually leans on — if it's dominated by `impressions_prev30`
alone, that's basically a fancier version of "big pages get flagged," worth saying plainly; (2)
whether errors are worse for a particular position bucket or a particular client — if the model
is much worse for small clients (fewer historical queries to learn from), that's a real
limitation to carry into ML-09 and the capstone, not something to bury.


In [13]:
import numpy as np
final_model = HistGradientBoostingRegressor(random_state=42)
final_model.fit(X, y)

try:
    importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("Feature importances:")
    print(importances)
except AttributeError:
    print("HistGradientBoostingRegressor doesn't expose feature_importances_ directly — "
          "use sklearn.inspection.permutation_importance instead:")
    from sklearn.inspection import permutation_importance
    r = permutation_importance(final_model, X, y, n_repeats=5, random_state=42)
    print(pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False))

# Error by position bucket — is the model worse for high-position (competitive) content?
pred_all = final_model.predict(X)
err = pd.DataFrame({"pos_bucket": df["pos_bucket_last30"].values,
                     "abs_error": np.abs(pred_all - y.values)})
print()
print("Mean absolute error by position bucket:")
print(err.groupby("pos_bucket").abs_error.mean().sort_values())


HistGradientBoostingRegressor doesn't expose feature_importances_ directly — use sklearn.inspection.permutation_importance instead:
impressions_prev30                 0.462177
rare_impressions_share             0.044190
content_visible_query_count        0.041729
avg_position_prev30                0.040967
char_count                         0.038627
rare_query_count                   0.034681
word_count                         0.032585
backlinks_was_missing              0.030246
query_char_count                   0.018842
search_volume                      0.016727
competition                        0.015970
clicks_prev30                      0.014901
query_token_count                  0.008893
cpc                                0.006541
backlinks                          0.005531
char_count_was_missing             0.004818
avg_position_prev30_was_missing    0.003304
search_volume_was_missing          0.000612
competition_was_missing            0.000000
cpc_was_missing                 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [14]:
print("ML-08 kernel test")
print("X exists:", "X" in globals())
print("y exists:", "y" in globals())
print("groups exists:", "groups" in globals())
print("X shape:", X.shape)

ML-08 kernel test
X exists: True
y exists: True
groups exists: True
X shape: (83310, 21)


In [15]:
# ============================================================
# FINAL VALIDATION — CORRECT PRECISION@50
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold


print("=" * 60)
print("FINAL VALIDATION CHECK")
print("=" * 60)


# ------------------------------------------------------------
# 1. Dataset sanity check
# ------------------------------------------------------------

print("\nDATASET")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique clients:", groups.nunique())
print("Feature count:", X.shape[1])


# ------------------------------------------------------------
# 2. Fold sizes
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GROUPED FOLD CHECK")
print("=" * 60)

gkf_final = GroupKFold(n_splits=5)

splits_final = list(
    gkf_final.split(X, y, groups=groups)
)

fold_rows = []

for fold, (tr, te) in enumerate(splits_final):

    train_clients = groups.iloc[tr].nunique()
    test_clients = groups.iloc[te].nunique()

    overlap = len(
        set(groups.iloc[tr]) &
        set(groups.iloc[te])
    )

    fold_rows.append({
        "fold": fold,
        "train_rows": len(tr),
        "test_rows": len(te),
        "train_clients": train_clients,
        "test_clients": test_clients,
        "client_overlap": overlap
    })

fold_check = pd.DataFrame(fold_rows)

print(fold_check.to_string(index=False))

print("\nMinimum test rows:", fold_check["test_rows"].min())
print("Maximum test rows:", fold_check["test_rows"].max())
print("Maximum client overlap:", fold_check["client_overlap"].max())


# ------------------------------------------------------------
# 3. Target tie check
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TARGET TIE CHECK")
print("=" * 60)

value_counts = y.value_counts()

tied_rows = value_counts[value_counts > 1].sum()
tie_fraction = tied_rows / len(y)

print("Total rows:", len(y))
print("Unique y values:", y.nunique())
print("Rows involved in ties:", tied_rows)
print("Tie fraction:", round(tie_fraction, 6))
print("Tie fraction (%):", round(tie_fraction * 100, 2))

print("\nMost frequent target values:")
print(value_counts.head(10))


# ------------------------------------------------------------
# 4. Define CORRECT Precision@50
# ------------------------------------------------------------
#
# Here we define "positive" as the top 50 TRUE target rows
# within each test fold.
#
# Precision@50 asks:
#
# Of the 50 rows selected by the model,
# how many belong to the true top-50?
#
# This is a ranking-retrieval metric.
# ------------------------------------------------------------

def precision_at_50_correct(predictions, y_true):

    predictions = np.asarray(predictions)
    y_true = np.asarray(y_true)

    k = min(50, len(y_true))

    pred_top50 = np.argsort(predictions)[::-1][:k]
    true_top50 = np.argsort(y_true)[::-1][:k]

    true_set = set(true_top50)

    hits = sum(
        idx in true_set
        for idx in pred_top50
    )

    return hits / k


# ------------------------------------------------------------
# 5. Run model vs baseline on SAME folds
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODEL VS BASELINE")
print("=" * 60)

model_scores = []
baseline_scores = []
auc_scores = []

for fold, (tr, te) in enumerate(splits_final):

    X_train = X.iloc[tr]
    X_test = X.iloc[te]

    y_train = y.iloc[tr]
    y_test = y.iloc[te]

    # -------------------------
    # Train model
    # -------------------------

    model = HistGradientBoostingRegressor(
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)


    # -------------------------
    # Baseline
    # -------------------------
    #
    # W05 already constructed:
    # prev30_ctr_gap
    #
    # Higher previous-window gap =
    # higher predicted refresh priority.
    # -------------------------

    baseline = (
        df["prev30_ctr_gap"]
        .iloc[te]
        .to_numpy()
    )


    # --------------------------------------------------------
    # Precision@50
    # --------------------------------------------------------

    model_p50 = precision_at_50_correct(
        pred,
        y_test
    )

    baseline_p50 = precision_at_50_correct(
        baseline,
        y_test
    )


    model_scores.append(model_p50)
    baseline_scores.append(baseline_p50)


    print(
        f"Fold {fold}: "
        f"test_rows={len(te):,} | "
        f"model P@50={model_p50:.4f} | "
        f"baseline P@50={baseline_p50:.4f}"
    )


# ------------------------------------------------------------
# 6. Results table
# ------------------------------------------------------------

results_final = pd.DataFrame({
    "fold": range(len(splits_final)),
    "model_precision_at_50": model_scores,
    "baseline_precision_at_50": baseline_scores
})


print("\n" + "=" * 60)
print("FINAL RESULTS TABLE")
print("=" * 60)

print(results_final.to_string(index=False))


# ------------------------------------------------------------
# 7. Mean results
# ------------------------------------------------------------

mean_model = np.mean(model_scores)
mean_baseline = np.mean(baseline_scores)

print("\n" + "=" * 60)
print("MEAN RESULTS")
print("=" * 60)

print(
    "Mean Model Precision@50:",
    round(mean_model, 4)
)

print(
    "Mean Baseline Precision@50:",
    round(mean_baseline, 4)
)

print(
    "Difference:",
    round(mean_model - mean_baseline, 4)
)


# ------------------------------------------------------------
# 8. Sanity checks
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SANITY CHECKS")
print("=" * 60)

if fold_check["client_overlap"].max() == 0:
    print("PASS: No client appears in both train and test.")

else:
    print("FAIL: Client overlap detected.")


if fold_check["test_rows"].min() >= 50:
    print("PASS: Every test fold has at least 50 rows.")

else:
    print("FAIL: A test fold has fewer than 50 rows.")


print("\nFINAL VALIDATION COMPLETE.")

FINAL VALIDATION CHECK

DATASET
X shape: (83310, 21)
y shape: (83310,)
Unique clients: 41
Feature count: 21

GROUPED FOLD CHECK
 fold  train_rows  test_rows  train_clients  test_clients  client_overlap
    0       63054      20256             40             1               0
    1       67544      15766             31            10               0
    2       67547      15763             32             9               0
    3       67547      15763             30            11               0
    4       67548      15762             31            10               0

Minimum test rows: 15762
Maximum test rows: 20256
Maximum client overlap: 0

TARGET TIE CHECK
Total rows: 83310
Unique y values: 10115
Rows involved in ties: 77504
Tie fraction: 0.930308
Tie fraction (%): 93.03

Most frequent target values:
ctr_gap_last30
-0.094598    540
-0.077931    483
-0.085507    461
-0.071521    458
-0.088587    453
-0.066026    445
-0.131444    435
-0.113587    433
-0.079496    430
-0.105709    429
N

In [16]:
# ============================================================
# TIE-AWARE RANKING DIAGNOSTIC — FINAL W05 CHECK
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor


print("=" * 60)
print("TIE-AWARE RANKING DIAGNOSTIC")
print("=" * 60)


# ------------------------------------------------------------
# 1. Tie-aware NDCG@50
# ------------------------------------------------------------

def ndcg_at_k(scores, y_true, k=50):
    """
    NDCG@K for continuous relevance scores.

    Higher y_true = higher refresh priority.
    Higher predicted score = model ranks row higher.
    """

    scores = np.asarray(scores, dtype=float)
    y_true = np.asarray(y_true, dtype=float)

    k = min(k, len(y_true))

    if k == 0:
        return np.nan

    # Rank rows according to predictions
    pred_order = np.argsort(scores)[::-1][:k]

    # Relevance of rows selected by the model
    relevance = y_true[pred_order]

    # Shift relevance so all gains are non-negative
    # This is necessary because ctr_gap_last30 can be negative.
    min_relevance = np.nanmin(y_true)

    relevance = relevance - min_relevance

    # Discount
    positions = np.arange(1, k + 1)

    discounts = np.log2(positions + 1)

    dcg = np.sum(
        (2 ** relevance - 1) / discounts
    )

    # Ideal ranking
    ideal_relevance = np.sort(y_true)[::-1][:k]

    ideal_relevance = ideal_relevance - min_relevance

    idcg = np.sum(
        (2 ** ideal_relevance - 1) / discounts
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


# ------------------------------------------------------------
# 2. Tie cutoff diagnostic
# ------------------------------------------------------------

def top50_cutoff_info(y_true):

    y_true = pd.Series(
        np.asarray(y_true)
    ).reset_index(drop=True)

    k = min(50, len(y_true))

    sorted_values = y_true.sort_values(
        ascending=False
    ).reset_index(drop=True)

    cutoff_value = sorted_values.iloc[k - 1]

    rows_above_cutoff = (y_true > cutoff_value).sum()

    rows_equal_cutoff = (y_true == cutoff_value).sum()

    rows_at_or_above = (
        y_true >= cutoff_value
    ).sum()

    return {
        "top50_cutoff": cutoff_value,
        "rows_above_cutoff": rows_above_cutoff,
        "rows_equal_cutoff": rows_equal_cutoff,
        "rows_at_or_above_cutoff": rows_at_or_above
    }


# ------------------------------------------------------------
# 3. Run client-grouped validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLIENT-GROUPED VALIDATION")
print("=" * 60)

diagnostic_results = []

for fold, (tr, te) in enumerate(splits_final):

    X_train = X.iloc[tr]
    X_test = X.iloc[te]

    y_train = y.iloc[tr]
    y_test = y.iloc[te]

    # --------------------------------------------------------
    # Train model
    # --------------------------------------------------------

    model = HistGradientBoostingRegressor(
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    model_pred = model.predict(X_test)


    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline_pred = (
        df["prev30_ctr_gap"]
        .iloc[te]
        .to_numpy()
    )


    y_test_array = y_test.to_numpy()


    # --------------------------------------------------------
    # Top-50 selections
    # --------------------------------------------------------

    k = min(50, len(y_test_array))

    model_top50_idx = np.argsort(
        model_pred
    )[::-1][:k]

    baseline_top50_idx = np.argsort(
        baseline_pred
    )[::-1][:k]


    # --------------------------------------------------------
    # Actual target values selected
    # --------------------------------------------------------

    model_selected_true = (
        y_test_array[model_top50_idx]
    )

    baseline_selected_true = (
        y_test_array[baseline_top50_idx]
    )


    # --------------------------------------------------------
    # Mean actual target among selected rows
    # --------------------------------------------------------

    model_mean_true = np.mean(
        model_selected_true
    )

    baseline_mean_true = np.mean(
        baseline_selected_true
    )


    # --------------------------------------------------------
    # Median actual target among selected rows
    # --------------------------------------------------------

    model_median_true = np.median(
        model_selected_true
    )

    baseline_median_true = np.median(
        baseline_selected_true
    )


    # --------------------------------------------------------
    # Maximum/minimum selected target
    # --------------------------------------------------------

    model_min_true = np.min(
        model_selected_true
    )

    baseline_min_true = np.min(
        baseline_selected_true
    )


    # --------------------------------------------------------
    # NDCG@50
    # --------------------------------------------------------

    model_ndcg = ndcg_at_k(
        model_pred,
        y_test_array,
        k=50
    )

    baseline_ndcg = ndcg_at_k(
        baseline_pred,
        y_test_array,
        k=50
    )


    # --------------------------------------------------------
    # True top-50 cutoff
    # --------------------------------------------------------

    cutoff = top50_cutoff_info(
        y_test_array
    )


    # --------------------------------------------------------
    # How many selected rows are exactly at cutoff?
    # --------------------------------------------------------

    model_cutoff_rows = np.sum(
        model_selected_true ==
        cutoff["top50_cutoff"]
    )

    baseline_cutoff_rows = np.sum(
        baseline_selected_true ==
        cutoff["top50_cutoff"]
    )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    diagnostic_results.append({

        "fold": fold,

        "test_rows": len(te),

        "test_clients": groups.iloc[te].nunique(),

        "top50_cutoff": cutoff["top50_cutoff"],

        "rows_above_cutoff":
            cutoff["rows_above_cutoff"],

        "rows_equal_cutoff":
            cutoff["rows_equal_cutoff"],

        "rows_at_or_above_cutoff":
            cutoff["rows_at_or_above_cutoff"],

        "model_mean_true_ctr_gap":
            model_mean_true,

        "baseline_mean_true_ctr_gap":
            baseline_mean_true,

        "model_median_true_ctr_gap":
            model_median_true,

        "baseline_median_true_ctr_gap":
            baseline_median_true,

        "model_min_true_ctr_gap":
            model_min_true,

        "baseline_min_true_ctr_gap":
            baseline_min_true,

        "model_cutoff_rows":
            model_cutoff_rows,

        "baseline_cutoff_rows":
            baseline_cutoff_rows,

        "model_ndcg_at_50":
            model_ndcg,

        "baseline_ndcg_at_50":
            baseline_ndcg
    })


# ------------------------------------------------------------
# 4. Display detailed results
# ------------------------------------------------------------

diagnostic_df = pd.DataFrame(
    diagnostic_results
)

print("\nDetailed fold results:")

print(
    diagnostic_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(
    "Mean model NDCG@50:",
    round(
        diagnostic_df[
            "model_ndcg_at_50"
        ].mean(),
        4
    )
)

print(
    "Mean baseline NDCG@50:",
    round(
        diagnostic_df[
            "baseline_ndcg_at_50"
        ].mean(),
        4
    )
)

print(
    "NDCG difference:",
    round(
        diagnostic_df[
            "model_ndcg_at_50"
        ].mean()
        -
        diagnostic_df[
            "baseline_ndcg_at_50"
        ].mean(),
        4
    )
)


print("\nMean actual ctr_gap among model top-50:")

print(
    round(
        diagnostic_df[
            "model_mean_true_ctr_gap"
        ].mean(),
        6
    )
)


print("\nMean actual ctr_gap among baseline top-50:")

print(
    round(
        diagnostic_df[
            "baseline_mean_true_ctr_gap"
        ].mean(),
        6
    )
)


print("\nDifference in mean selected true ctr_gap:")

print(
    round(
        diagnostic_df[
            "model_mean_true_ctr_gap"
        ].mean()
        -
        diagnostic_df[
            "baseline_mean_true_ctr_gap"
        ].mean(),
        6
    )
)


# ------------------------------------------------------------
# 6. Cutoff tie summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOP-50 CUTOFF TIE SUMMARY")
print("=" * 60)

print(
    diagnostic_df[
        [
            "fold",
            "top50_cutoff",
            "rows_above_cutoff",
            "rows_equal_cutoff",
            "rows_at_or_above_cutoff",
            "model_cutoff_rows",
            "baseline_cutoff_rows"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 7. Interpretation helpers
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("INTERPRETATION CHECKS")
print("=" * 60)

mean_model_ndcg = diagnostic_df[
    "model_ndcg_at_50"
].mean()

mean_baseline_ndcg = diagnostic_df[
    "baseline_ndcg_at_50"
].mean()

mean_model_true = diagnostic_df[
    "model_mean_true_ctr_gap"
].mean()

mean_baseline_true = diagnostic_df[
    "baseline_mean_true_ctr_gap"
].mean()

mean_cutoff_ties = diagnostic_df[
    "rows_equal_cutoff"
].mean()


print(
    f"Average number of rows tied at top-50 cutoff: "
    f"{mean_cutoff_ties:.2f}"
)

print(
    f"Model NDCG@50: "
    f"{mean_model_ndcg:.4f}"
)

print(
    f"Baseline NDCG@50: "
    f"{mean_baseline_ndcg:.4f}"
)

print(
    f"Model selected-row mean true ctr_gap: "
    f"{mean_model_true:.6f}"
)

print(
    f"Baseline selected-row mean true ctr_gap: "
    f"{mean_baseline_true:.6f}"
)

print("\nDONE.")

TIE-AWARE RANKING DIAGNOSTIC

CLIENT-GROUPED VALIDATION

Detailed fold results:
 fold  test_rows  test_clients  top50_cutoff  rows_above_cutoff  rows_equal_cutoff  rows_at_or_above_cutoff  model_mean_true_ctr_gap  baseline_mean_true_ctr_gap  model_median_true_ctr_gap  baseline_median_true_ctr_gap  model_min_true_ctr_gap  baseline_min_true_ctr_gap  model_cutoff_rows  baseline_cutoff_rows  model_ndcg_at_50  baseline_ndcg_at_50
    0      20256             1      0.010102                 49                  1                       50                 0.001634                   -0.062375                   0.001889                     -0.043029               -0.012623                  -0.488587                  0                     0          0.988150             0.897696
    1      15766            10      0.008857                 49                  1                       50                 0.003094                   -0.042114                   0.004751                     -0.035431     